In [0]:
dbutils.widgets.dropdown(name='Reset', defaultValue='True', choices=['True', 'False'], label="Reset: Drop previous table")

# Create a Unity Catalog Volume
spark.sql("CREATE VOLUME IF NOT EXISTS porya_catalog.default.cv_clf")
# Sets the file path variable 
volume_file_path = "/Volumes/porya_catalog/default/cv_clf"

In [0]:
# Define variables expected from global-setup
base_path = f"{volume_file_path}/intel_image_clf/raw_images"
train_dir = f"{base_path}/seg_train/seg_train"
valid_dir = f"{base_path}/seg_test/seg_test"
outcomes = ["buildings", "sea", "glacier", "forest", "street", "mountain"]
main_dir_2write = volume_file_path

In [0]:
from pyspark.sql import functions as f

delta_train_name = "train_imgs_main.delta"
delta_val_name = "valid_imgs_main.delta"

if dbutils.widgets.get('Reset') == 'True':
    dbutils.fs.rm(f"{volume_file_path}/{delta_train_name}", recurse=True)
    dbutils.fs.rm(f"{volume_file_path}/{delta_val_name}", recurse=True)


def prep_data2delta(
    dir_name,
    outcomes,
    name2write,
    path2write="YOUR_PATH",
    write2delta=True,
    returnDF=False,
):

    mapping_dict = {
        "buildings": 0,
        "sea": 1,
        "glacier": 2,
        "forest": 3,
        "street": 4,
        "mountain": 5,
    }
    all_dfs = []
    # As we have multi label problem we will loop over labels to save them all under 1 main training set
    for label_name in outcomes:
        df = (
            spark.read.format("binaryfile")
            .option("recursiveFileLookup", "true")
            .load(f"{dir_name}/{label_name}")
            .withColumn("label_name", f.lit(label_name))
            .withColumn("label_id", f.lit(mapping_dict[label_name]))
            .withColumn("image_name", f.element_at(f.split(f.col("path"), "/"), -1))
            .withColumn(
                "id", f.split(f.col("image_name"), "\\.jpg").getItem(0).astype("int")
            )
        )
        if write2delta:
            df.write.format("delta").mode("append").save(f"{path2write}/{name2write}")
        if returnDF:
            all_dfs.append(df)

    if returnDF and all_dfs:
        result = all_dfs[0]
        for additional_df in all_dfs[1:]:
            result = result.union(additional_df)
        return result
    
prep_data2delta(
    train_dir,
    outcomes,
    delta_train_name,
    write2delta=True,
    path2write=main_dir_2write,
    returnDF=None,
)

# COMMAND ----------

prep_data2delta(
    valid_dir,
    outcomes,
    delta_val_name,
    path2write=main_dir_2write,
    write2delta=True,
    returnDF=None,
)